# Python 中的线程、进程与协程

本笔记汇总三种常见并发方式的特点与基本用法，并提供最小可运行示例。

## 核心概念概览
- **线程 (thread)**：运行在同一进程内，共享内存，创建开销小，适合 I/O 密集任务；受 GIL 影响，CPU 密集型加速有限。
- **进程 (process)**：拥有独立内存空间，避免 GIL 影响，适合 CPU 密集任务；创建和通信开销较大。
- **协程 (coroutine)**：用户态的轻量级任务，通过事件循环合作式切换；非常适合大量 I/O 等待场景，需要 `async/await` 风格的库支持。

选择小技巧：
- I/O 密集、依赖阻塞库 → 线程。
- CPU 密集 → 进程。
- I/O 密集、支持异步库 → 协程。

## 线程示例：并行处理 I/O 密集任务
使用 `threading.Thread` 启动多个线程，并通过 `queue.Queue` 汇总结果。

In [ ]:
import threading
import queue
import time

work_items = [1, 2, 3, 4]
results = queue.Queue()

def io_like_task(x: int):
    time.sleep(0.2)  # 模拟 I/O 等待
    results.put((x, x * x))

threads = [threading.Thread(target=io_like_task, args=(n,)) for n in work_items]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("线程结果 (顺序与提交顺序无关)：")
while not results.empty():
    print(results.get())

## 进程示例：利用多核运行 CPU 密集计算
`multiprocessing` 通过多进程绕过 GIL，每个进程有独立地址空间。

In [ ]:
from multiprocessing import Pool

def cpu_heavy(x: int) -> int:
    # 简单的 CPU 密集型计算
    return sum(i * i for i in range(x))

if __name__ == "__main__":
    with Pool() as pool:
        inputs = [10_000, 20_000, 30_000]
        outputs = pool.map(cpu_heavy, inputs)
    print("进程结果：", list(zip(inputs, outputs)))

## 协程示例：事件循环驱动的异步 I/O
使用 `asyncio` 创建协程任务，在等待 I/O 时自动切换，提高并发度。

In [ ]:
import asyncio

async def fake_io(name: str, delay: float):
    await asyncio.sleep(delay)
    return f"{name} 完成"

async def main():
    tasks = [asyncio.create_task(fake_io("task A", 0.5)),
             asyncio.create_task(fake_io("task B", 0.2)),
             asyncio.create_task(fake_io("task C", 0.1))]
    for coro in asyncio.as_completed(tasks):
        print(await coro)

asyncio.run(main())

## 小结
- 在同一项目中可以混合使用多种并发方式，根据任务类型选择合适方案。
- 如果需要在 Jupyter Notebook 中运行多进程示例，确保使用 `if __name__ == "__main__":` 保护，以避免在子进程重复导入时出错。
- 异步代码依赖协程友好的库（例如 `aiohttp`、`asyncpg`），阻塞式库需要通过线程池或进程池封装后再在协程中使用。